### Face 2: ingenieria de caracteristicas.

| Vamos a extraer caracteristicas por cada registro

Importamos librerias

In [7]:
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
tqdm.pandas()

In [8]:
df = pd.read_csv('dataset_optimizado.csv')
def crear_features_pro(df):
    df = df.copy()
    
    df['Hora_Inicio'] = pd.to_datetime(
        df['Hora_Inicio'],
        format='%H:%M:%S'
    )
    df['Hora_Minutos'] = df['Hora_Inicio'].dt.hour * 60 + df['Hora_Inicio'].dt.minute
    
    cond_pico = (
        ((df['Hora_Minutos'] >= 480) & (df['Hora_Minutos'] <= 540)) |
        ((df['Hora_Minutos'] >= 720) & (df['Hora_Minutos'] <= 780))
    )
    df['Es_Hora_Pico'] = cond_pico.astype(int)
    
    df['Periodo_Dia'] = pd.cut(df['Hora_Minutos'], 
                                bins=[0, 360, 720, 1080, 1440], 
                                labels=[0, 1, 2, 3]) 
    
    df = df.sort_values(['Direccion', 'Hora_Inicio'])
    
    group = df.groupby('Direccion')
    
    df['Total_Vehiculos_lag1'] = group['Total_Vehiculos'].shift(1)
    df['Ocupacion_lag1'] = group['Ocupacion_Espacial_%'].shift(1)
    
    df['Media_Movil_3ciclos'] = group['Total_Vehiculos'].transform(lambda x: x.rolling(3, min_periods=1).mean())
    
    df['Tendencia_Vehiculos'] = group['Total_Vehiculos'].diff()
    
    df['Capacidad_Teorica'] = df['move_time_s'] / df['Tiempo_Medio_s'].replace(0, 1)
    df['Saturacion_Actual'] = df['Total_Vehiculos'] / df['Capacidad_Teorica'].replace(0, 1)
    
    df['Cluster_Hora_Pico'] = (df['cluster'] * 10) + df['Es_Hora_Pico']
    
    df = df.fillna(0)
    
    return df

df = crear_features_pro(df)
display(df.head())
df.to_csv('completo_clusters_con_features.csv', index=False)

,Total_Vehiculos,Tiempo_Medio_s,Ocupacion_Espacial_%,Hora_Inicio,Hora_Fin,Dia_Semana,Direccion,move_time_s,cluster,tiempo_optimo_s,...,Hora_Minutos,Es_Hora_Pico,Periodo_Dia,Total_Vehiculos_lag1,Ocupacion_lag1,Media_Movil_3ciclos,Tendencia_Vehiculos,Capacidad_Teorica,Saturacion_Actual,Cluster_Hora_Pico
1523,7,1.67,17.71,1900-01-01 08:00:09,08:00:42,2,1,33,0,35,...,480,1,1,0.0,0.00,7.000000,0.0,19.760479,0.354242,1
2284,9,3.68,21.95,1900-01-01 08:00:20,08:00:53,3,1,33,0,35,...,480,1,1,7.0,17.71,8.000000,2.0,8.967391,1.003636,1
3,7,4.23,19.05,1900-01-01 08:00:29,08:01:02,4,1,33,0,35,...,480,1,1,9.0,21.95,7.666667,-2.0,7.801418,0.897273,1
764,7,2.00,16.25,1900-01-01 08:01:32,08:02:05,1,1,33,0,35,...,481,1,1,7.0,19.05,7.666667,0.0,16.500000,0.424242,1
3045,9,3.03,23.80,1900-01-01 08:01:43,08:02:16,5,1,33,0,35,...,481,1,1,7.0,16.25,7.666667,2.0,10.891089,0.826364,1
